# Phase 4.1 - Aggressive Model Compression

**Author:** Rafael  **Dataset:** Tiny-ImageNet-200 (200 classes, 64x64 RGB)

Push the best Phase 1-3 models below INT8. PyTorch-native quantization
strategies, evaluated on the same axis so bit-widths are directly comparable:

| Method | Bits | Mode | Models |
|---|---|---|---|
| INT8 | W8/A8 | PTQ (calibrate) | all 5 (anchor) |
| INT4 PTQ | W4/A8 | PTQ (calibrate) | all 5 |
| INT4 QAT | W4/A8 | retrain | 3 tiny |
| Mixed | W4-8/A8 per-layer | retrain | 3 tiny |
| INT2 PTQ | W2/A8 | PTQ (calibrate) | all 5 |
| Ternary (TWN) | `{-α,0,+α}` weight-only | PTQ (no calib) | all 5 |
| Binary (BWN) | `{-α,+α}` weight-only | PTQ (no calib) | all 5 |

INT4/mixed/INT2 use **FakeQuantize simulation** (fbgemm has no INT4 kernel): accuracy is
faithful, size is the theoretical packed size (`theoretical_size_mb`). Ternary/binary
quantize weights directly (`apply_weight_ptq`), activations FP32. All eval on GPU.

**Targets:** `mobilenetv2`, `alexnet_bottleneck`, `alexnet_fire`,
`alexnet_depthwisesep` (QAT-unstable edge case), `alexnet_final_fire_residual`.

Layout: 1. Imports 2. Data 3. FP32 baselines 4. Compression helpers
5. INT8 6. INT4 PTQ 7. INT4 QAT 8. Mixed precision 8b. Sub-INT4 (INT2/ternary/binary) 9. Comparison & summaries

## 1. Imports & reproducibility

In [ ]:
import json
import os
import random
import sys
import time
from dataclasses import asdict, replace
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchinfo
import wandb

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from ml import (
    DataConfig, TrainerConfig, QATConfig,
    create_imagenet_loaders,
    Trainer, make_qat_callback, auto_resume_path, compress_checkpoint,
    load_best_model, build_comparison_table, create_results_summary,
    disk_mb, gzip_mb, compute_flops,
    make_qconfig, prepare_sim, calibrate,
    compute_layer_sensitivity, assign_mixed_precision, apply_weight_qat, theoretical_size_mb,
)
from ml import find_fuse_groups
from configs.loader import load_config

from models import (
    MobileNetV2TV, AlexNetBottleneck, AlexNetFire,
    AlexNetDepthwiseSep, AlexNetFinalFireResidual,
)

torch.backends.quantized.engine = "fbgemm"

In [3]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SAVE_DIR    = project_root / "checkpoints" / "compression_phase4_1"
RESULTS_DIR = project_root / "results" / "compression_phase4_1"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

data_cfg = DataConfig(**load_config("data.yaml"))
data_cfg.seed = SEED
fp32_cfg = TrainerConfig(**load_config("training.yaml"))
comp_cfg = load_config("compression.yaml")
print(device)

cuda


## 2. Dataset & loaders

Standard train/val loaders plus a small **calibration loader** (single-worker, no shuffle) used for PTQ range collection and sensitivity analysis.

In [4]:
import kagglehub

dataset_path = kagglehub.dataset_download("akash2sharma/tiny-imagenet")
train_path = os.path.join(dataset_path, "tiny-imagenet-200", "train")
data_cfg.dataset_path = train_path

train_ds, val_ds, train_loader, val_loader = create_imagenet_loaders(data_cfg)

calib_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=data_cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True,
)

print(f"Train {len(train_ds):,} | Val {len(val_ds):,} | classes {data_cfg.num_classes}")
print(f"Batches: train={len(train_loader)} val={len(val_loader)}")

Train 90,000 | Val 10,000 | classes 200
Batches: train=1406 val=157


## 3. Target models & FP32 baselines

The 5 targets live in three different Phase 1-3 checkpoint dirs. We load each
FP32 `*_best.pth`, evaluate it, and record its baseline (accuracy, disk size, params, MACs).

In [5]:
TARGETS = {
    "mobilenetv2":                 dict(ctor=MobileNetV2TV,            src="baselines_qat_phase1"),
    "alexnet_bottleneck":          dict(ctor=AlexNetBottleneck,        src="compensation_phase3"),
    "alexnet_fire":                dict(ctor=AlexNetFire,              src="compensation_phase3"),
    "alexnet_depthwisesep":        dict(ctor=AlexNetDepthwiseSep,      src="compensation_phase3"),
    "alexnet_final_fire_residual": dict(ctor=AlexNetFinalFireResidual, src="final_architecture_phase4"),
}
ALL_MODELS  = list(TARGETS)
TINY_MODELS = ["alexnet_bottleneck", "alexnet_fire", "alexnet_depthwisesep"]

IMG = data_cfg.img_size
INPUT_SIZE = (1, 3, IMG, IMG)


def load_fp32(name):
    """Load the best FP32 checkpoint for a target from its source phase dir."""
    spec = TARGETS[name]
    src_dir = project_root / "checkpoints" / spec["src"]
    return load_best_model(name, spec["ctor"], src_dir, device, eval_mode=True)


def eval_on(model, loader):
    """Top-1/Top-5/loss via Trainer.evaluate (device follows model)."""
    t = Trainer(model, train_loader, loader, replace(fp32_cfg, use_amp=False),
                next(model.parameters()).device, SAVE_DIR, "eval",
                num_classes=data_cfg.num_classes)
    return t.evaluate(topk=(1, 5))

In [ ]:
fp32_baseline = {}
for name in ALL_MODELS:
    m = load_fp32(name)
    info = torchinfo.summary(m, input_size=INPUT_SIZE, verbose=0)
    ev = eval_on(m, val_loader)
    flops = compute_flops(m.cpu().eval(), input_size=INPUT_SIZE)
    src_dir = project_root / "checkpoints" / TARGETS[name]["src"]
    compress_checkpoint(src_dir / f"{name}_best.pth")
    fp32_baseline[name] = {
        "top1": ev["top1"], "top5": ev["top5"],
        "params_m": info.total_params / 1e6,
        "disk_mb": disk_mb(src_dir / f"{name}_best.pth"),
        "gzip_mb": gzip_mb(src_dir / f"{name}_best.pth"),
        "size_mb_32": theoretical_size_mb(m, w_bits=32),
        "macs": flops["macs"],
    }
    print(f"{name:30s} top1={ev['top1']:.2f}% top5={ev['top5']:.2f}% "
          f"params={info.total_params/1e6:.2f}M size={fp32_baseline[name]['disk_mb']:.1f}MB")
    del m; torch.cuda.empty_cache()

## 4. Compression helpers

`fuse_map_for` auto-detects Conv-BN(-ReLU) groups. `run_ptq` calibrates a
fake-quant model (no retraining). `run_qat` fine-tunes one. Every result is
appended to `ROWS` in the shared schema and mirrored to a per-model record.

In [7]:
ROWS = []
per_model = {n: {"fp32_baseline": fp32_baseline[n], "methods": []} for n in ALL_MODELS}


def fuse_map_for(name):
    """Best-effort auto fuse map (find_fuse_groups recurses into .features etc.)."""
    return find_fuse_groups(TARGETS[name]["ctor"]())


def _load_best_into(model, run_name):
    ckpt = torch.load(SAVE_DIR / f"{run_name}_best.pth", map_location=str(device), weights_only=False)
    model.load_state_dict(ckpt.get("model_state_dict", ckpt))
    return model


def record(name, method, precision, ev, size_mb, bench, notes,
           train_hrs=None, calib_min=None, bits_map=None):
    base = fp32_baseline[name]
    ratio = base["size_mb_32"] / size_mb if size_mb else None
    row = {
        "model": name, "method": method, "precision": precision,
        "fp32_baseline_acc": base["top1"], "compressed_top1_acc": ev["top1"],
        "top5_acc": ev["top5"], "acc_drop_pp": base["top1"] - ev["top1"],
        "compression_ratio": ratio,
        "fp32_size_mb": base["size_mb_32"], "compressed_size_mb": size_mb,
        "latency_ms": bench["latency_ms_per_image"],
        "throughput_img_s": bench["throughput_img_per_s"],
        "params_m": base["params_m"], "acc_per_mb": ev["top1"] / size_mb if size_mb else None,
        "training_time_hrs": train_hrs, "calibration_time_min": calib_min,
        "top1_%": ev["top1"], "top5_%": ev["top5"], "size_MB": size_mb,
        "notes": notes,
    }
    ROWS.append(row)
    rec = dict(row); rec["bits_map"] = bits_map
    per_model[name]["methods"].append(rec)
    print(f"  -> {method:12s} top1={ev['top1']:.2f}% ({row['acc_drop_pp']:+.2f}pp) "
          f"size={size_mb:.2f}MB ratio={ratio:.1f}x")


def wandb_run(name, method, cfg_extra):
    return wandb.init(
        project="alexnet-compression", group="phase-4-1-compression",
        name=f"{name}_{method}",
        config={"arch": name, "method": method, "phase": "4.1",
                "dataset": "tiny-imagenet-200", **cfg_extra},
        tags=["phase-4-1", method, name, "tiny-imagenet", "compression"],
        mode="offline", reinit=True,
    )

In [8]:
def run_ptq(name, method, w_bits, a_bits, n_calib, precision, notes):
    """Post-training: fuse -> fake-quant -> calibrate -> eval. No retraining."""
    model = load_fp32(name)
    qconfig = make_qconfig(w_bits=w_bits, a_bits=a_bits, per_channel=True)
    sim = prepare_sim(model, fuse_map_for(name), qconfig, device)
    t0 = time.perf_counter()
    sim = calibrate(sim, calib_loader, device, n_samples=n_calib)
    calib_min = (time.perf_counter() - t0) / 60
    ev = eval_on(sim, val_loader)
    bench = Trainer(sim, train_loader, val_loader, replace(fp32_cfg, use_amp=False),
                    device, SAVE_DIR, "bench", num_classes=data_cfg.num_classes).benchmark(warmup=50)
    size_mb = theoretical_size_mb(model, w_bits=w_bits)
    run = wandb_run(name, method, {"w_bits": w_bits, "a_bits": a_bits, "calib_samples": n_calib})
    run.summary.update({"top1": ev["top1"], "top5": ev["top5"], "size_mb": size_mb})
    wandb.finish()
    record(name, method, precision, ev, size_mb, bench, notes, calib_min=calib_min)
    del model, sim; torch.cuda.empty_cache()


def run_qat(name, method, w_bits, a_bits, mcfg, precision, notes, bits_map=None, sim=None):
    """QAT: fine-tune a fake-quant (or mixed) model, reload best, eval."""
    model = load_fp32(name)
    if sim is None:
        qconfig = make_qconfig(w_bits=w_bits, a_bits=a_bits, per_channel=mcfg.get("per_channel", True))
        sim = prepare_sim(model, fuse_map_for(name), qconfig, device)
    run_name = f"{method}_{name}"
    qat_cfg = replace(fp32_cfg, epochs=mcfg["epochs"], lr=mcfg["lr"],
                      weight_decay=mcfg["weight_decay"], use_amp=False)
    cb = make_qat_callback(mcfg["freeze_bn_epoch"], mcfg["disable_observer_epoch"])
    resume = auto_resume_path(SAVE_DIR, run_name)

    if (SAVE_DIR / f"{run_name}_best.pth").exists() and not resume:
        print(f"  (reusing {run_name}_best.pth)")
    else:
        run = wandb_run(name, method, {"w_bits": w_bits, "a_bits": a_bits, **mcfg})
        tr = Trainer(sim, train_loader, val_loader, qat_cfg, device, SAVE_DIR, run_name,
                     num_classes=data_cfg.num_classes, wandb_run=run, epoch_callback=cb,
                     log_file=SAVE_DIR / f"{run_name}.log")
        fit = tr.fit(resume_from=resume)
        wandb.finish()
        per_model[name].setdefault("train_time_hrs", {})[method] = fit["total_training_time_s"] / 3600

    sim = _load_best_into(sim, run_name).eval()
    ev = eval_on(sim, val_loader)
    bench = Trainer(sim, train_loader, val_loader, qat_cfg, device, SAVE_DIR, "bench",
                    num_classes=data_cfg.num_classes).benchmark(warmup=50)
    size_mb = theoretical_size_mb(model, w_bits=w_bits, bits_map=bits_map)
    train_hrs = per_model[name].get("train_time_hrs", {}).get(method)
    record(name, method, precision, ev, size_mb, bench, notes,
           train_hrs=train_hrs, bits_map=bits_map)
    del model, sim; torch.cuda.empty_cache()

## 5. INT8 (anchor)

Uniform fake-quant W8/A8 PTQ across all 5 models - the known-good reference point. (Real INT8-QAT numbers from Phases 1-3 remain valid cross-checks.)

In [9]:
for name in ALL_MODELS:
    print(name)
    run_ptq(name, "int8", w_bits=8, a_bits=8,
            n_calib=comp_cfg["int4_ptq"]["calibration_samples"],
            precision="int8", notes="fake-quant W8/A8 PTQ")

mobilenetv2


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


size_mb,2.36518
top1,55.41303
top5,79.93391


  -> int8         top1=55.41% (+2.58pp) size=2.37MB ratio=4.0x
alexnet_bottleneck


size_mb,0.36716
top1,44.23525
top5,70.65533


...

  -> int8         top1=43.94% (+0.04pp) size=0.49MB ratio=4.0x
alexnet_depthwisesep


size_mb,0.29911
top1,43.65471
top5,69.48452


  -> int8         top1=43.65% (+0.74pp) size=0.30MB ratio=4.0x
alexnet_final_fire_residual


size_mb,0.6663
top1,49.75736
top5,74.84932


  -> int8         top1=49.76% (+0.04pp) size=0.67MB ratio=4.0x


## 6. INT4 PTQ

Weights to 4-bit post-training (per-channel), activations INT8. The fast low-bit baseline.

In [10]:
for name in ALL_MODELS:
    print(name)
    run_ptq(name, "int4_ptq", w_bits=4, a_bits=8,
            n_calib=comp_cfg["int4_ptq"]["calibration_samples"],
            precision="int4", notes="fake-quant W4/A8 PTQ, per-channel")

mobilenetv2


size_mb,1.19895
top1,3.5026
top5,11.59575


  -> int4_ptq     top1=3.50% (+54.49pp) size=1.20MB ratio=7.8x
alexnet_bottleneck


size_mb,0.18536
top1,35.32445
top5,62.18264


  -> int4_ptq     top1=35.32% (+9.30pp) size=0.19MB ratio=7.9x
alexnet_fire


...

  -> int4_ptq     top1=37.72% (+6.26pp) size=0.25MB ratio=7.9x
alexnet_depthwisesep


size_mb,0.15161
top1,9.99649
top5,25.41479


  -> int4_ptq     top1=10.00% (+34.39pp) size=0.15MB ratio=7.8x
alexnet_final_fire_residual


size_mb,0.3354
top1,40.01107
top5,66.9007


  -> int4_ptq     top1=40.01% (+9.78pp) size=0.34MB ratio=7.9x


## 7. INT4 QAT

Retrain under 4-bit weight fake-quant on the 3 tiny models - does training recover the PTQ gap? (Watch **alexnet_depthwisesep**, the -2.92 pp QAT edge case.)

In [11]:
mcfg_int4 = comp_cfg["int4_qat"]
for name in TINY_MODELS:
    print(name)
    run_qat(name, "int4_qat", w_bits=4, a_bits=8, mcfg=mcfg_int4,
            precision="int4", notes="fake-quant W4/A8 QAT, per-channel")

alexnet_bottleneck



================= Run Summary =================
Model          : int4_qat_alexnet_bottleneck
Epochs         : 20
Best Val Top-1 : 44.46%
Best Val Top-5 : 70.71%
Final Val Top-1: 40.83%
Final Val Top-5: 67.64%
Best Val Loss  : inf
Total Time     : 00:00:00


  -> int4_qat     top1=44.34% (+0.29pp) size=0.19MB ratio=7.9x
alexnet_fire


Validation: 100%|██████████| 157/157 [00:05<00:00, 28.02it/s, loss=3.0319, top1=42.99%, top5=69.62%]
Epoch  20/20 | train_loss=3.0276 train_acc=42.18% | val_loss=3.0319 val_acc=42.99% val_top5=69.62% | lr=0.00e+00 peak_mem=1058MB time=119.3s
Early stopping at epoch 20

================= Run Summary =================
Model          : int4_qat_alexnet_fire
Epochs         : 20
Best Val Top-1 : 44.41%
Best Val Top-5 : 70.60%
Final Val Top-1: 42.99%
Final Val Top-5: 69.62%
Best Val Loss  : inf
Total Time     : 00:01:59


epoch_time_s,▁
lr,▁
peak_gpu_mem_mb,▁
train_acc,▁
train_loss,▁
val_acc,▁
val_loss,▁
val_top5,▁
epoch_time_s,119.34136
lr,0
peak_gpu_mem_mb,1058.11768


  -> int4_qat     top1=44.26% (-0.28pp) size=0.25MB ratio=7.9x
alexnet_depthwisesep


Validation: 100%|██████████| 157/157 [00:01<00:00, 89.14it/s, loss=3.6106, top1=30.32%, top5=55.82%] 
Epoch  20/20 | train_loss=3.2590 train_acc=36.51% | val_loss=3.6106 val_acc=30.32% val_top5=55.82% | lr=0.00e+00 peak_mem=341MB time=38.5s
Early stopping at epoch 20

================= Run Summary =================
Model          : int4_qat_alexnet_depthwisesep
Epochs         : 20
Best Val Top-1 : 41.38%
Best Val Top-5 : 67.37%
Final Val Top-1: 30.32%
Final Val Top-5: 55.82%
Best Val Loss  : inf
Total Time     : 00:00:38


epoch_time_s,▁
lr,▁
peak_gpu_mem_mb,▁
train_acc,▁
train_loss,▁
val_acc,▁
val_loss,▁
val_top5,▁
epoch_time_s,38.54068
lr,0
peak_gpu_mem_mb,341.05615


  -> int4_qat     top1=41.32% (+3.07pp) size=0.15MB ratio=7.8x


## 8. Mixed precision (INT4/INT8 per-layer)

Rank layers by INT4 sensitivity, keep the top `int8_ratio` at INT8 and the rest at INT4, then fine-tune. Tests whether per-layer mixing beats uniform INT4.

In [12]:
import copy as _copy
mcfg_mix = comp_cfg["mixed_precision"]
for name in TINY_MODELS:
    print(name)
    model = load_fp32(name)
    base = _copy.deepcopy(model).to(device).train()
    # sensitivity + per-layer qconfig on the UNFUSED graph so layer names match,
    # THEN fuse (carries each conv's qconfig into the fused module) and prepare_qat.
    sens = compute_layer_sensitivity(base, calib_loader, device, w_bits=4)
    bits_map = assign_mixed_precision(base, sens, int8_ratio=mcfg_mix["int8_ratio"],
                                      per_channel=mcfg_mix["per_channel"])
    base.qconfig = make_qconfig(8, 8, mcfg_mix["per_channel"])  # default for unmarked mods
    fm = fuse_map_for(name)
    if fm:
        try:
            torch.ao.quantization.fuse_modules_qat(base, fm, inplace=True)
        except Exception as e:
            print("  fusion skipped:", e)
    base.train()  # fuse_modules_qat flips to eval; prepare_qat requires train mode
    torch.ao.quantization.prepare_qat(base, inplace=True)
    n8 = sum(v == 8 for v in bits_map.values()); n4 = sum(v == 4 for v in bits_map.values())
    print(f"  layers: INT8={n8} INT4={n4}")
    run_qat(name, "mixed", w_bits=4, a_bits=8, mcfg=mcfg_mix,
            precision="int4/8", notes=f"per-layer INT4/INT8 ({n8} hi / {n4} lo)",
            bits_map=bits_map, sim=base)
    del model, base; torch.cuda.empty_cache()

alexnet_bottleneck
  layers: INT8=5 INT4=11


Validation: 100%|██████████| 157/157 [00:02<00:00, 67.90it/s, loss=2.9410, top1=44.44%, top5=71.02%]
Epoch  11/20 | train_loss=2.9800 train_acc=42.51% | val_loss=2.9410 val_acc=44.44% val_top5=71.02% | lr=4.22e-06 peak_mem=498MB time=47.9s
Early stopping at epoch 11

================= Run Summary =================
Model          : mixed_alexnet_bottleneck
Epochs         : 11
Best Val Top-1 : 44.77%
Best Val Top-5 : 70.69%
Final Val Top-1: 44.44%
Final Val Top-5: 71.02%
Best Val Loss  : inf
Total Time     : 00:00:47


epoch_time_s,▁
lr,▁
peak_gpu_mem_mb,▁
train_acc,▁
train_loss,▁
val_acc,▁
val_loss,▁
val_top5,▁
epoch_time_s,47.86551
lr,0.0
peak_gpu_mem_mb,497.87451


  -> mixed        top1=44.61% (+0.01pp) size=0.21MB ratio=6.9x
alexnet_fire
  layers: INT8=5 INT4=11


Validation: 100%|██████████| 157/157 [00:02<00:00, 57.19it/s, loss=3.0225, top1=43.11%, top5=69.96%]
Epoch  12/20 | train_loss=3.1171 train_acc=39.75% | val_loss=3.0225 val_acc=43.11% val_top5=69.96% | lr=3.45e-06 peak_mem=1069MB time=93.2s
Early stopping at epoch 12

================= Run Summary =================
Model          : mixed_alexnet_fire
Epochs         : 12
Best Val Top-1 : 44.79%
Best Val Top-5 : 70.95%
Final Val Top-1: 43.11%
Final Val Top-5: 69.96%
Best Val Loss  : inf
Total Time     : 00:01:33


epoch_time_s,▁
lr,▁
peak_gpu_mem_mb,▁
train_acc,▁
train_loss,▁
val_acc,▁
val_loss,▁
val_top5,▁
epoch_time_s,93.21594
lr,0.0
peak_gpu_mem_mb,1068.89648


  -> mixed        top1=44.70% (-0.72pp) size=0.29MB ratio=6.7x
alexnet_depthwisesep
  layers: INT8=3 INT4=8



================= Run Summary =================
Model          : mixed_alexnet_depthwisesep
Epochs         : 20
Best Val Top-1 : 44.35%
Best Val Top-5 : 69.43%
Final Val Top-1: 44.16%
Final Val Top-5: 69.39%
Best Val Loss  : inf
Total Time     : 00:00:00


  -> mixed        top1=44.27% (+0.12pp) size=0.16MB ratio=7.5x


## 8b. Sub-INT4: INT2, ternary (TWN) & binary (BWN) QAT

How far can retraining push these below INT8? Three ever-more-aggressive weight
schemes across **all 5 models**, all **quantization-aware trained** (PTQ alone
collapses this aggressively — Section 6 already shows that gap for INT4, no
need to re-demonstrate it at 1-2 bits):

| Method | Weight levels | Bits | Activations |
|---|---|---|---|
| INT2 QAT | 16 uniform | W2 | A8 (fake-quant) |
| Ternary (TWN) QAT | `{-α, 0, +α}` | ~2 | FP32 |
| Binary (BWN) QAT | `{-α, +α}` | 1 | FP32 |

- **INT2** rides the existing fake-quant path (`run_qat`, W2/A8) — same recipe as Section 7's INT4 QAT.
- **Ternary** (Ternary Weight Networks): per-channel threshold `Δ = 0.7·mean|w|`, scale `α = mean|w|` over survivors — the "(-1, 0, 1)" scheme.
- **Binary** (XNOR-Net BWN): `w → sign(w)·α`, `α = mean|w|` per channel.

Ternary/binary are **weight-only** (`apply_weight_qat`), the setup those papers
report — activations stay FP32. Training uses a straight-through estimator (STE):
the forward pass quantizes the weight, the backward pass passes gradient through
unchanged to a latent FP32 weight, which the optimizer actually updates.

In [13]:
def run_weight_qat(name, method, scheme, w_bits_size, mcfg, precision, notes):
    """Weight-only binary/ternary QAT (STE). Activations FP32; retrained."""
    model = load_fp32(name)
    sim = apply_weight_qat(model, scheme).to(device).train()
    run_name = f"{method}_{name}"
    qat_cfg = replace(fp32_cfg, epochs=mcfg["epochs"], lr=mcfg["lr"],
                      weight_decay=mcfg["weight_decay"], use_amp=False)
    resume = auto_resume_path(SAVE_DIR, run_name)

    if (SAVE_DIR / f"{run_name}_best.pth").exists() and not resume:
        print(f"  (reusing {run_name}_best.pth)")
    else:
        run = wandb_run(name, method, {"scheme": scheme, "w_bits": w_bits_size, **mcfg})
        tr = Trainer(sim, train_loader, val_loader, qat_cfg, device, SAVE_DIR, run_name,
                     num_classes=data_cfg.num_classes, wandb_run=run,
                     log_file=SAVE_DIR / f"{run_name}.log")
        fit = tr.fit(resume_from=resume)
        wandb.finish()
        per_model[name].setdefault("train_time_hrs", {})[method] = fit["total_training_time_s"] / 3600

    sim = _load_best_into(sim, run_name).eval()
    ev = eval_on(sim, val_loader)
    bench = Trainer(sim, train_loader, val_loader, qat_cfg, device, SAVE_DIR, "bench",
                    num_classes=data_cfg.num_classes).benchmark(warmup=50)
    size_mb = theoretical_size_mb(model, w_bits=w_bits_size)
    train_hrs = per_model[name].get("train_time_hrs", {}).get(method)
    record(name, method, precision, ev, size_mb, bench, notes, train_hrs=train_hrs)
    del model, sim; torch.cuda.empty_cache()


for name in ALL_MODELS:
    print(name)
    run_qat(name, "int2_qat", w_bits=2, a_bits=8, mcfg=comp_cfg["int2_qat"],
            precision="int2", notes="fake-quant W2/A8 QAT, per-channel")
    run_weight_qat(name, "ternary_qat", "ternary", w_bits_size=2, mcfg=comp_cfg["ternary_qat"],
                   precision="ternary", notes="weight-only TWN QAT (STE), per-channel, A=FP32")
    run_weight_qat(name, "binary_qat", "binary", w_bits_size=1, mcfg=comp_cfg["binary_qat"],
                   precision="binary", notes="weight-only BWN QAT (STE), per-channel, A=FP32")

mobilenetv2


Validation: 100%|██████████| 157/157 [00:03<00:00, 46.80it/s, loss=5.6332, top1=0.93%, top5=3.32%]
wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch   8/20 | train_loss=5.2607 train_acc=1.90% | val_loss=5.6332 val_acc=0.93% val_top5=3.32% | lr=6.55e-06 peak_mem=712MB time=63.7s
Validation: 100%|██████████| 157/157 [00:03<00:00, 48.63it/s, loss=5.7487, top1=0.51%, top5=2.65%]
Epoch   9/20 | train_loss=5.1956 train_acc=2.19% | val_loss=5.7487 val_acc=0.51% val_top5=2.65% | lr=5.78e-06 peak_mem=676MB time=66.9s
Validation: 100%|██████████| 157/157 [00:03<00:00, 46.61it/s, loss=5.6554, top1=0.66%, top5=3.15%]
Epoch  10/20 | train_loss=5.1548 train_acc=2.43% | val_loss=5.6554 val_acc=0.66% val_top5=3.15% | lr=5.00e-06 peak_mem=676MB time=64.1s
Validation: 100%|██████

best_val_acc,▁
epoch_time_s,▁█▂▂▁▁
lr,█▇▅▄▂▁
peak_gpu_mem_mb,█▁▁▁▁▁
train_acc,▁▃▅▆▇█
train_loss,█▅▄▃▂▁
val_acc,█▁▄▂▂▂
val_loss,▅▇▅█▁▅
val_top5,█▁▆▂▂▆
best_val_acc,0.93
epoch_time_s,63.84111


  -> int2_qat     top1=0.96% (+57.04pp) size=0.62MB ratio=15.2x


Validation: 100%|██████████| 157/157 [00:02<00:00, 74.42it/s, loss=5.1667, top1=2.78%, top5=9.58%]
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch   1/20 | train_loss=5.3957 train_acc=1.44% | val_loss=5.1667 val_acc=2.78% val_top5=9.58% | lr=9.94e-06 peak_mem=486MB time=39.5s
Validation: 100%|██████████| 157/157 [00:02<00:00, 62.26it/s, loss=5.0025, top1=3.99%, top5=13.36%]
Epoch   2/20 | train_loss=5.1879 train_acc=2.18% | val_loss=5.0025 val_acc=3.99% val_top5=13.36% | lr=9.76e-06 peak_mem=486MB time=40.8s
Validation: 100%|██████████| 157/157 [00:02<00:00, 74.81it/s, loss=4.8700, top1=5.19%, top5=17.13%]
Epoch   3/20 | train_loss=5.0796 train_acc=2.93% | val_loss=4.8700 val_acc=5.19% val_top5=17.13% | lr=9.46e-06 peak_mem=486MB time=45.4s
Validation: 100%|██████████| 157/157 [00:02<00:00, 69.06it/s, loss=4.7758, top1=6.21%, top5=19.99%]
Epoch   4/20 | train_loss=4.9914 train_acc=3.70% | val_loss=4.7758 val_acc=6.21% val_top5=

...

best_val_acc,▁▄▅▆▇▇████
epoch_time_s,█▅▆▇▇▆▇▇▇▆█▇▇█▆█▅▃▁▂
lr,███▇▇▇▆▆▅▅▄▃▃▂▂▂▁▁▁▁
peak_gpu_mem_mb,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▃▅▆▆▇▇▇▇███████████
train_loss,█▅▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▄▅▆▇▇▇▇▇████▇██▇███
val_loss,█▅▄▃▂▂▂▂▂▁▁▁▁▂▁▁▁▁▁▁
val_top5,▁▄▆▆▇▇▇▇▇████▇██████
best_val_acc,36.59
epoch_time_s,25.28899


  -> ternary_qat  top1=36.54% (+13.26pp) size=0.17MB ratio=15.6x


Validation: 100%|██████████| 157/157 [00:01<00:00, 151.52it/s, loss=5.3571, top1=2.27%, top5=9.48%]
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch   1/20 | train_loss=5.7841 train_acc=1.40% | val_loss=5.3571 val_acc=2.27% val_top5=9.48% | lr=9.94e-06 peak_mem=222MB time=23.0s
Validation: 100%|██████████| 157/157 [00:01<00:00, 138.83it/s, loss=5.0095, top1=4.25%, top5=15.06%]
Epoch   2/20 | train_loss=5.3164 train_acc=2.51% | val_loss=5.0095 val_acc=4.25% val_top5=15.06% | lr=9.76e-06 peak_mem=222MB time=22.5s
Validation: 100%|██████████| 157/157 [00:01<00:00, 144.65it/s, loss=4.8150, top1=6.09%, top5=19.62%]
Epoch   3/20 | train_loss=5.0891 train_acc=3.70% | val_loss=4.8150 val_acc=6.09% val_top5=19.62% | lr=9.46e-06 peak_mem=222MB time=22.6s
Validation: 100%|██████████| 157/157 [00:01<00:00, 137.71it/s, loss=4.6816, top1=7.79%, top5=24.18%]
Epoch   4/20 | train_loss=4.9486 train_acc=4.92% | val_loss=4.6816 val_acc=7.79% val_t

best_val_acc,▁▂▃▄▄▅▆▆▆▆▇▇███
epoch_time_s,█▆▆▃▃▅▃▃▃▂▄▃▂▃▁▂▁▂▄▅
lr,███▇▇▇▆▆▅▅▄▃▃▂▂▂▁▁▁▁
peak_gpu_mem_mb,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▂▃▃▄▅▅▆▆▇▇▇▇███████
train_loss,█▆▄▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val_acc,▁▂▃▄▄▅▆▆▆▆▆▇▇▇██████
val_loss,█▆▅▄▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁
val_top5,▁▂▃▄▅▆▆▆▆▇▇▇▇███████
best_val_acc,16.89
epoch_time_s,22.4452


  -> binary_qat   top1=16.92% (+32.87pp) size=0.09MB ratio=30.4x


## 9. Comparison table, per-model summaries & experiment summary

In [14]:
df = build_comparison_table(ROWS)
cols = ["model","method","precision","fp32_baseline_acc","compressed_top1_acc","acc_drop_pp",
        "compression_ratio","compressed_size_mb","latency_ms","acc_per_mb","notes"]
df_out = df[[c for c in cols if c in df.columns]].copy()
df.to_csv(RESULTS_DIR / "final_comparison.csv", index=False)

by_method = (df.groupby("method")
               .agg(mean_top1=("compressed_top1_acc","mean"),
                    mean_drop_pp=("acc_drop_pp","mean"),
                    mean_ratio=("compression_ratio","mean"),
                    mean_acc_per_mb=("acc_per_mb","mean"),
                    n=("model","count"))
               .reset_index())
by_method.to_csv(RESULTS_DIR / "compression_by_method.csv", index=False)

for name, rec in per_model.items():
    with open(RESULTS_DIR / f"{name}_compression_summary.json", "w") as f:
        json.dump(rec, f, indent=2, default=str)

df_out

,model,method,precision,fp32_baseline_acc,compressed_top1_acc,acc_drop_pp,compression_ratio,compressed_size_mb,latency_ms,acc_per_mb,notes
0,alexnet_final_fire_residual,binary_qat,binary,49.793291,16.922915,32.870376,30.402817,0.087219,0.083508,194.027319,"weight-only BWN QAT (STE), per-channel, A=FP32"
1,alexnet_fire,binary_qat,binary,43.976474,15.851358,28.125116,30.579463,0.064100,0.143487,247.290046,"weight-only BWN QAT (STE), per-channel, A=FP32"
2,alexnet_bottleneck,binary_qat,binary,44.622979,12.624389,31.998590,29.755547,0.049000,0.102137,257.641714,"weight-only BWN QAT (STE), per-channel, A=FP32"
3,alexnet_depthwisesep,binary_qat,binary,44.390011,4.683565,39.706446,28.896348,0.040979,0.108463,114.292419,"weight-only BWN QAT (STE), per-channel, A=FP32"
4,mobilenetv2,binary_qat,binary,57.995582,3.561827,54.433754,28.871847,0.324280,0.124406,10.983809,"weight-only BWN QAT (STE), per-channel, A=FP32"
5,alexnet_final_fire_residual,int2_qat,int2,49.793291,28.976917,20.816374,15.603367,0.169945,0.133692,170.507856,"fake-quant W2/A8 QAT, per-channel"
6,alexnet_fire,int2_qat,int2,43.976474,27.147138,16.829336,15.648263,0.125263,0.246888,216.720753,"fake-quant W2/A8 QAT, per-channel"
7,alexnet_bottleneck,int2_qat,int2,44.622979,20.460297,24.162681,15.436591,0.094452,0.134241,216.621330,"fake-quant W2/A8 QAT, per-channel"
8,alexnet_depthwisesep,int2_qat,int2,44.390011,9.482220,34.907791,15.209547,0.077855,0.102341,121.793532,"fake-quant W2/A8 QAT, per-channel"
9,mobilenetv2,int2_qat,int2,57.995582,0.958424,57.037158,15.202976,0.615837,0.295502,1.556294,"fake-quant W2/A8 QAT, per-channel"


In [15]:
def dominated(r, others):
    for o in others:
        if (o["compressed_top1_acc"] >= r["compressed_top1_acc"]
                and o["compressed_size_mb"] <= r["compressed_size_mb"]
                and o["latency_ms"] <= r["latency_ms"]
                and (o["compressed_top1_acc"] > r["compressed_top1_acc"]
                     or o["compressed_size_mb"] < r["compressed_size_mb"]
                     or o["latency_ms"] < r["latency_ms"])):
            return True
    return False

recs = df.to_dict("records")
pareto = [r for r in recs if not dominated(r, recs)]
pareto_df = pd.DataFrame(pareto).sort_values("compressed_top1_acc", ascending=False)
pareto_df[[c for c in cols if c in pareto_df.columns]].to_csv(RESULTS_DIR / "pareto_frontier.csv", index=False)
print(f"Pareto-optimal: {len(pareto)} of {len(recs)}")
pareto_df[["model","method","compressed_top1_acc","compressed_size_mb","latency_ms"]]

Pareto-optimal: 15 of 31


,model,method,compressed_top1_acc,compressed_size_mb,latency_ms
9,mobilenetv2,int8,55.413032,2.365181,0.318327
10,alexnet_final_fire_residual,int8,49.757361,0.666298,0.198092
6,alexnet_fire,mixed,44.696173,0.292267,0.331698
7,alexnet_bottleneck,mixed,44.609120,0.209892,0.300873
4,alexnet_bottleneck,int4_qat,44.337183,0.185356,0.190940
8,alexnet_depthwisesep,mixed,44.269145,0.157833,0.094335
5,alexnet_depthwisesep,int4_qat,41.324723,0.151607,0.123792
11,alexnet_final_fire_residual,ternary_qat,36.537331,0.169945,0.090011
12,alexnet_fire,ternary_qat,32.809520,0.125263,0.157997
13,alexnet_bottleneck,ternary_qat,31.351751,0.094452,0.118686


In [16]:
create_results_summary(
    results={
        "phase": "4.1", "title": "Aggressive Model Compression",
        "models": ALL_MODELS,
        "methods": ["int8", "int4_ptq", "int4_qat", "mixed", "int2_ptq", "ternary_ptq", "binary_ptq"],
        "fp32_baseline": fp32_baseline,
        "rows": ROWS,
        "by_method": by_method.to_dict("records"),
        "pareto": pareto_df[["model","method","compressed_top1_acc","compressed_size_mb"]].to_dict("records"),
    },
    config=fp32_cfg,
    output_path=RESULTS_DIR / "experiment_summary.json",
)
print("Saved:", RESULTS_DIR / "experiment_summary.json")

Saved: /home/rafael/Documents/alexnet_rafael/results/compression_phase4_1/experiment_summary.json


## W&B - syncing offline runs

All runs logged offline under project **`alexnet-compression`**, group
**`phase-4-1-compression`**, phase tag `4.1`. Sync when ready:

```bash
wandb sync --sync-all
```